In [1]:
import transformers
import torch
import time
import statistics

# Local model directory
model_dir = "/workspace/models/Llama-3.1-8B-Instruct/"

print(f"Loading model from: {model_dir}")
tokenizer = transformers.AutoTokenizer.from_pretrained(model_dir)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)

# Long text to summarize
long_text = """
Artificial intelligence (AI) has evolved from narrow task-specific systems to large-scale
foundation models capable of handling diverse cognitive activities. Over the last decade,
transformer architectures have revolutionized language processing by enabling context-aware
representation learning through self-attention. Models like GPT, PaLM, and Llama have scaled
to hundreds of billions of parameters, demonstrating emergent abilities such as reasoning,
abstraction, and multi-step planning. However, challenges remain in ensuring factual accuracy,
energy efficiency, and alignment with human values. As AI systems become integrated into
healthcare, education, and law, questions of bias, accountability, and interpretability grow
more urgent. Researchers are exploring retrieval-augmented generation, instruction tuning,
and reinforcement learning from human feedback (RLHF) to improve controllability. The next
frontier lies in multimodal intelligence — unifying text, vision, audio, and action models
into a coherent reasoning agent that can operate safely and transparently in complex real-world
environments.
""" * 5

# Create proper chat messages
messages = [
    {"role": "system", "content": "You are an expert summarizer."},
    {"role": "user", "content": f"Summarize the following text in concise form:\n\n{long_text}"}
]

# Apply chat template to convert messages → valid text prompt
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Warm-up (excluded from timing)
print("Running warm-up inference...")
_ = pipeline(prompt, max_new_tokens=256, temperature=0.3, do_sample=False)

# Timed benchmark
NUM_RUNS = 10
times = []

print(f"\nStarting {NUM_RUNS} summarization runs...\n")
for i in range(NUM_RUNS):
    start = time.time()
    outputs = pipeline(prompt, max_new_tokens=256, temperature=0.3, do_sample=False)
    end = time.time()
    elapsed = end - start
    times.append(elapsed)
    print(f"Run {i+1}/{NUM_RUNS}: {elapsed:.3f}s")

# Compute stats
avg_time = statistics.mean(times)
p95 = statistics.quantiles(times, n=100)[94]
total = sum(times)

print("\n=== Timing Results ===")
print(f"Average inference time: {avg_time:.3f}s")
print(f"95th percentile latency: {p95:.3f}s")
print(f"Total time: {total:.2f}s")

# Show clean summary output
print("\n=== Summary ===")
print(outputs[0]["generated_text"].strip())


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading model from: /workspace/models/Llama-3.1-8B-Instruct/


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.34it/s]
Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running warm-up inference...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Run 7/10: 2.208s


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Run 8/10: 2.211s


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Run 9/10: 2.206s
Run 10/10: 2.208s

=== Timing Results ===
Average inference time: 2.210s
95th percentile latency: 2.217s
Total time: 22.10s

=== Summary ===
Here is a concise summary of the text:

Artificial intelligence (AI) has evolved from narrow task-specific systems to large-scale foundation models capable of handling diverse cognitive activities. Transformer architectures have revolutionized language processing through self-attention, enabling models like GPT, PaLM, and Llama to demonstrate emergent abilities such as reasoning, abstraction, and multi-step planning. However, challenges remain in ensuring factual accuracy, energy efficiency, and alignment with human values. Researchers are exploring methods to improve controllability, including retrieval-augmented generation, instruction tuning, and reinforcement learning from human feedback. The next frontier lies in multimodal intelligence, unifying text, vision, audio, and action models into a coherent reasoning agent that can 